In [3]:
!pip install transformers torch pandas numpy scikit-learn openpyxl langdetect

     ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
     - -------------------------------------- 0.3/10.4 MB 1.5 MB/s eta 0:00:07
     --- ------------------------------------ 0.8/10.4 MB 2.0 MB/s eta 0:00:05
     ---- ----------------------------------- 1.1/10.4 MB 2.3 MB/s eta 0:00:05
     ------ --------------------------------- 1.7/10.4 MB 2.3 MB/s eta 0:00:04
     -------- ------------------------------- 2.3/10.4 MB 2.8 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2.9 MB/s eta 0:00:03
     ----------- ---------------------------- 3.1/10.4 MB 2

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 90, in read
    data = self.__fp.read(amt)
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 466, in read
    s = self.fp.read(amt)
  File "C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\socket.py", line 705, in readinto
    

In [4]:
import pandas as pd
import numpy as np
import json
import ast
import re
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================
# CONFIGURATION
# ============================================

class Config:
    # Model
    MODEL_NAME = "xlm-roberta-base"  # or "xlm-roberta-large" if GPU memory permits
    MAX_LEN = 256
    BATCH_SIZE = 16
    EPOCHS = 5
    LR = 2e-5
    DROPOUT = 0.2
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    
    # Data paths
    TRAIN_PATH = "data/DeepX_train.xlsx"
    VAL_PATH = "data/DeepX_validation.xlsx"
    UNLABELED_PATH = "data/DeepX_unlabeled.xlsx"
    OUTPUT_PATH = "output/predictions.json"
    MODEL_SAVE_PATH = "output/best_model.pt"
    
    # Aspects and sentiments (including 'none' as required by the data)
    ASPECTS = ['food', 'service', 'price', 'cleanliness', 
               'delivery', 'ambiance', 'app_experience', 'general', 'none']
    SENTIMENTS = ['positive', 'negative', 'neutral', 'none']
    
    # Inference threshold (can be tuned on validation set)
    THRESHOLD = 0.5
    
    # Random seed
    SEED = 42


In [ ]:

# ============================================
# FRANKO PROCESSING (Arabic with Latin letters)
# ============================================

FRANKO_MAP = {
    '2': 'ء', '3': 'ع', '4': 'ش', '5': 'خ', '6': 'ط', '7': 'ح', '8': 'غ', '9': 'ق',
    'a': 'ا', 'b': 'ب', 'd': 'د', 'f': 'ف', 'g': 'ج', 'h': 'ه', 'j': 'ج', 'k': 'ك',
    'l': 'ل', 'm': 'م', 'n': 'ن', 'p': 'ب', 'q': 'ق', 'r': 'ر', 's': 'س', 't': 'ت',
    'v': 'ڤ', 'w': 'و', 'x': 'خ', 'y': 'ي', 'z': 'ز', 'th': 'ث', 'kh': 'خ', 'sh': 'ش'
}

def is_franko(text):
    """Detect if text is Franko (Arabic written with Latin letters and numbers)"""
    if pd.isna(text):
        return False
    text = str(text)
    # If it already has Arabic characters, not Franko
    if re.search(r'[\u0600-\u06FF]', text):
        return False
    # Franko typically contains numbers used as Arabic letters
    return bool(re.search(r'[2346789]', text.lower()))

def franko_to_arabic(text):
    """Convert Franko text to Arabic script"""
    if not is_franko(text):
        return text
    result = str(text)
    # Sort by length descending to handle multi-character patterns first
    for eng, arb in sorted(FRANKO_MAP.items(), key=lambda x: -len(x[0])):
        result = re.sub(re.escape(eng), arb, result, flags=re.IGNORECASE)
    return result

def preprocess_text(text, star_rating=None):
    """Complete preprocessing pipeline for any language review"""
    if pd.isna(text):
        text = ""
    text = str(text)
    
    # Convert Franko to Arabic
    if is_franko(text):
        text = franko_to_arabic(text)
    
    # Basic cleaning
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    text = re.sub(r'([!?.])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Add star rating as a hint for the model
    if star_rating is not None:
        text = f"[STAR{star_rating}] " + text
    
    return text if text else "no text"



In [ ]:

# ============================================
# LABEL ENCODER
# ============================================

class LabelEncoder:
    """Encode aspect-sentiment pairs to multi-label binary vectors"""
    
    def __init__(self, aspects, sentiments):
        self.aspects = aspects
        self.sentiments = sentiments
        self.n_aspects = len(aspects)
        self.n_sentiments = len(sentiments)
        self.n_classes = self.n_aspects * self.n_sentiments
        self.aspect_to_idx = {a: i for i, a in enumerate(aspects)}
        self.sentiment_to_idx = {s: i for i, s in enumerate(sentiments)}
    
    def encode(self, aspect_sentiments):
        """Convert dict to multi-label binary vector"""
        labels = np.zeros(self.n_classes, dtype=np.float32)
        if not isinstance(aspect_sentiments, dict):
            return labels
        for aspect, sentiment in aspect_sentiments.items():
            if aspect in self.aspect_to_idx and sentiment in self.sentiment_to_idx:
                idx = self.aspect_to_idx[aspect] * self.n_sentiments + self.sentiment_to_idx[sentiment]
                labels[idx] = 1.0
        return labels
    
    def decode(self, logits, threshold=0.5):
        """Convert model output to aspect_sentiments dict"""
        if isinstance(logits, torch.Tensor):
            probs = torch.sigmoid(logits).cpu().numpy()
        else:
            probs = 1 / (1 + np.exp(-logits))
        
        result = {}
        for i, p in enumerate(probs):
            if p > threshold:
                aspect_idx = i // self.n_sentiments
                sentiment_idx = i % self.n_sentiments
                aspect = self.aspects[aspect_idx]
                sentiment = self.sentiments[sentiment_idx]
                # Skip the default 'none' aspect with 'none' sentiment
                if aspect == 'none' and sentiment == 'none':
                    continue
                result[aspect] = sentiment
        return result
    
    def decode_batch(self, logits_batch, threshold=0.5):
        """Decode a batch of predictions"""
        results = []
        for logits in logits_batch:
            results.append(self.decode(logits, threshold))
        return results


In [ ]:

# ============================================
# DATASET
# ============================================

class AspectDataset(Dataset):
    """PyTorch dataset for aspect sentiment classification"""
    
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32)
        }



In [ ]:

# ============================================
# MODEL ARCHITECTURE
# ============================================

class AspectSentimentModel(nn.Module):
    """Multi-lingual model for aspect sentiment classification"""
    
    def __init__(self, model_name, num_classes, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits



In [ ]:

# ============================================
# METRICS
# ============================================

def compute_metrics(predictions, labels, threshold=0.5):
    """Compute multi-label classification metrics"""
    pred_binary = (predictions > threshold).astype(int)
    
    macro_f1 = f1_score(labels, pred_binary, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, pred_binary, average='micro', zero_division=0)
    weighted_f1 = f1_score(labels, pred_binary, average='weighted', zero_division=0)
    
    return {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'weighted_f1': weighted_f1
    }



In [ ]:

# ============================================
# DATA LOADING
# ============================================

def load_data(filepath, label_encoder, has_labels=True):
    """Load and preprocess data from Excel or CSV file"""
    print(f"Loading {filepath}...")
    filepath = str(filepath)
    if filepath.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath, encoding='cp1252')
    print(f"  Loaded {len(df)} samples")
    
    # Handle missing values
    df['review_text'] = df['review_text'].fillna('')
    df['star_rating'] = df['star_rating'].fillna(3).astype(int)
    
    texts = []
    stars = []
    labels = [] if has_labels else None
    
    for _, row in df.iterrows():
        text = preprocess_text(row['review_text'], row['star_rating'])
        texts.append(text)
        stars.append(row['star_rating'])
        
        if has_labels:
            # Parse aspect_sentiments column
            aspect_sent = row.get('aspect_sentiments', {})
            if isinstance(aspect_sent, str):
                try:
                    aspect_sent = ast.literal_eval(aspect_sent)
                except (ValueError, SyntaxError):
                    aspect_sent = {}
            elif pd.isna(aspect_sent):
                aspect_sent = {}
            labels.append(label_encoder.encode(aspect_sent))

    

    return df, texts, stars, labels

In [ ]:

# ============================================
# TRAINING
# ============================================

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    metrics = compute_metrics(all_preds, all_labels)
    metrics['loss'] = total_loss / len(loader)
    
    return metrics


def evaluate(model, loader, device, threshold=0.5):
    """Evaluate model on validation set"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(batch['labels'].numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    metrics = compute_metrics(all_preds, all_labels, threshold)
    
    return metrics, all_preds, all_labels


In [ ]:

# ============================================
# INFERENCE
# ============================================

def predict(model, texts, tokenizer, label_encoder, device, max_len, batch_size, threshold=0.5):
    """Run inference on a list of texts"""
    model.eval()
    all_predictions = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encodings = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        
        with torch.no_grad():
            logits = model(
                encodings['input_ids'].to(device),
                encodings['attention_mask'].to(device)
            )
        
        for logit in logits:
            pred = label_encoder.decode(logit, threshold)
            all_predictions.append(pred)
    
    return all_predictions


In [ ]:

# ============================================
# MAIN PIPELINE
# ============================================

def setup_device_and_seeds(config):
    """Setup device and random seeds"""
    torch.manual_seed(config.SEED)
    np.random.seed(config.SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    return device


def find_best_threshold(model, val_loader, label_encoder, device, thresholds=None):
    """Find optimal threshold for multi-label classification"""
    if thresholds is None:
        thresholds = np.arange(0.3, 0.7, 0.05)
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(batch['labels'].numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    best_f1 = 0
    best_thresh = 0.5
    
    for thresh in thresholds:
        pred_binary = (all_preds > thresh).astype(int)
        macro_f1 = f1_score(all_labels, pred_binary, average='macro', zero_division=0)
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_thresh = thresh
    
    print(f"Best threshold: {best_thresh:.2f} (F1: {best_f1:.4f})")
    return best_thresh, best_f1


def save_predictions(predictions, df, output_path):
    """Save predictions in the required JSON format"""
    output = []
    for i, pred in enumerate(predictions):
        review_id = df.iloc[i]['review_id']
        output.append({
            "review_id": int(review_id),
            "aspects": list(pred.keys()),
            "aspect_sentiments": pred
        })
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {len(output)} predictions to {output_path}")
    return output


def print_statistics(predictions):
    """Print statistics about predictions"""
    all_aspects = [a for p in predictions for a in p['aspects']]
    all_sentiments = [s for p in predictions for s in p['aspect_sentiments'].values()]
    
    print("\n=== Prediction Statistics ===")
    print(f"Total predictions: {len(predictions)}")
    print(f"Total aspect mentions: {len(all_aspects)}")
    
    print("\nAspect distribution:")
    for aspect, count in Counter(all_aspects).most_common():
        print(f"  {aspect}: {count} ({count/len(all_aspects)*100:.1f}%)")
    
    print("\nSentiment distribution:")
    for sentiment, count in Counter(all_sentiments).most_common():
        print(f"  {sentiment}: {count} ({count/len(all_sentiments)*100:.1f}%)")
    
    empty = sum(1 for p in predictions if len(p['aspects']) == 0)
    print(f"\nReviews with no aspects: {empty} ({empty/len(predictions)*100:.1f}%)")
    
    multi = sum(1 for p in predictions if len(p['aspects']) >= 2)
    print(f"Reviews with 2+ aspects: {multi} ({multi/len(predictions)*100:.1f}%)")


In [ ]:
torch.cuda.is_available()

In [ ]:
import subprocess
import sys

print("="*50)
print("GPU DIAGNOSTIC")
print("="*50)

# Check nvidia-smi
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ NVIDIA drivers installed")
        # Extract CUDA version
        for line in result.stdout.split('\n'):
            if 'CUDA Version' in line:
                print(f"  {line.strip()}")
    else:
        print("✗ NVIDIA drivers NOT installed or not in PATH")
except FileNotFoundError:
    print("✗ nvidia-smi not found - NVIDIA drivers not installed")

# Check PyTorch
try:
    import torch
    print(f"\nPyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("\nPyTorch was installed WITHOUT CUDA support")
        print("Run: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
except ImportError:
    print("✗ PyTorch not installed")

print("\n" + "="*50)

In [ ]:
# ============================================
# MAIN FUNCTION - KAGGLE VERSION
# ============================================

def main():
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)

    # Setup device and random seeds
    device = setup_device_and_seeds(Config)

    # Initialize label encoder
    label_encoder = LabelEncoder(Config.ASPECTS, Config.SENTIMENTS)
    print(f"Number of classes: {label_encoder.n_classes}")

    # Load data (reading CSV files from Config paths)
    train_df, train_texts, _, train_labels = load_data(
        Config.TRAIN_PATH, label_encoder, has_labels=True
    )
    val_df, val_texts, _, val_labels = load_data(
        Config.VAL_PATH, label_encoder, has_labels=True
    )
    unlabeled_df, unlabeled_texts, _, _ = load_data(
        Config.UNLABELED_PATH, label_encoder, has_labels=False
    )

    print(f"\nTraining samples: {len(train_texts)}")
    print(f"Validation samples: {len(val_texts)}")
    print(f"Unlabeled samples: {len(unlabeled_texts)}")

    # Initialize tokenizer
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

    # Create datasets and dataloaders
    train_dataset = AspectDataset(train_texts, train_labels, tokenizer, Config.MAX_LEN)
    val_dataset = AspectDataset(val_texts, val_labels, tokenizer, Config.MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

    # Initialize model
    model = AspectSentimentModel(
        Config.MODEL_NAME,
        label_encoder.n_classes,
        Config.DROPOUT
    ).to(device)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Training
    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)

    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    total_steps = len(train_loader) * Config.EPOCHS
    num_warmup = int(total_steps * Config.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup,
        num_training_steps=total_steps
    )
    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0

    for epoch in range(Config.EPOCHS):
        train_metrics = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_metrics, _, _ = evaluate(model, val_loader, device)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}")
        print(f"  Train Loss: {train_metrics['loss']:.4f}, F1: {train_metrics['macro_f1']:.4f}")
        print(f"  Val F1: macro={val_metrics['macro_f1']:.4f}, micro={val_metrics['micro_f1']:.4f}")

        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            torch.save(model.state_dict(), Config.MODEL_SAVE_PATH)
            print(f"  -> Saved best model (F1={best_f1:.4f})")

    # Load best model
    model.load_state_dict(torch.load(Config.MODEL_SAVE_PATH))

    # Find best threshold on validation set
    best_thresh, _ = find_best_threshold(model, val_loader, label_encoder, device)
    Config.THRESHOLD = best_thresh

    # Final evaluation on validation set
    print("\n" + "="*60)
    print("FINAL EVALUATION ON VALIDATION SET")
    print("="*60)

    val_metrics, val_preds, val_labels_arr = evaluate(model, val_loader, device, Config.THRESHOLD)
    print(f"Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"Micro F1: {val_metrics['micro_f1']:.4f}")
    print(f"Weighted F1: {val_metrics['weighted_f1']:.4f}")

    # Show sample predictions
    print("\nSample predictions (first 5 validation samples):")
    for i in range(min(5, len(val_texts))):
        true_dict = label_encoder.decode(val_labels_arr[i])
        pred_dict = label_encoder.decode(val_preds[i], Config.THRESHOLD)
        print(f"\n  Review: {val_texts[i][:80]}...")
        print(f"    True: {true_dict}")
        print(f"    Pred: {pred_dict}")

    # Inference on unlabeled data
    print("\n" + "="*60)
    print("INFERENCE ON UNLABELED DATA")
    print("="*60)

    predictions = predict(
        model, unlabeled_texts, tokenizer, label_encoder, device,
        Config.MAX_LEN, Config.BATCH_SIZE, Config.THRESHOLD
    )

    # Save predictions
    output = save_predictions(predictions, unlabeled_df, Config.OUTPUT_PATH)

    # Print statistics
    print_statistics(output)

    # Show sample
    print("\n=== Sample Prediction ===")
    print(json.dumps(output[0], ensure_ascii=False, indent=2))

    print("\n" + "="*60)
    print("DONE!")
    print("="*60)


if __name__ == "__main__":
    # Use local data files from the repo
    local_train = Path("data/DeepX_train.xlsx")
    if local_train.exists():
        Config.TRAIN_PATH = str(local_train)
        Config.VAL_PATH = str(Path("data/DeepX_validation.xlsx"))
        Config.UNLABELED_PATH = str(Path("data/DeepX_unlabeled.xlsx"))
        Config.OUTPUT_PATH = str(Path("output/predictions.json"))
        Config.MODEL_SAVE_PATH = str(Path("output/best_model.pt"))
        print("Using local repo data file paths.")
    else:
        print("Local data files not found; using configured paths.")

    Path("output").mkdir(parents=True, exist_ok=True)

    device = setup_device_and_seeds(Config)
    print(f"Running on: {device}")
    print(f"Train: {Config.TRAIN_PATH}")
    print(f"Val: {Config.VAL_PATH}")
    print(f"Test: {Config.UNLABELED_PATH}")
    print(f"Output: {Config.OUTPUT_PATH}")
    print(f"Model: {Config.MODEL_NAME}")
    print(f"Epochs: {Config.EPOCHS}")
    print(f"Batch size: {Config.BATCH_SIZE}")
    print("="*60)

    main()


In [ ]:
import sys
# Remove the -f argument that Kaggle adds automatically
if '-f' in sys.argv:
    sys.argv.remove('-f')
    
# Remove any .json kernel file arguments
for arg in sys.argv[:]:
    if arg.endswith('.json') and '/kernel-' in arg:
        sys.argv.remove(arg)

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False
